# Embrace Plus — Derived Modalities Generator
Generates date-wise files for: **PR, RESP, ACT_COUNT, ACT_INT, MET** from existing raw modalities
(accelerometer, bvp, eda, steps, systolicPeaks, temperature).

Output files match your existing schema: `datetime_utc, <value column>, datetime_ist, account_number, date`
(same columns as your `eda`/`accelerometer` files), with `datetime_ist = datetime_utc + 5:30`.

`ACT_CLASS` is **not** computed here — it needs a Capture24-trained model (separate step).

Confirmed real schema:
- `eda`: `datetime_utc, eda, datetime_ist, account_number, date`
- `accelerometer`: `datetime_utc, accelerometer_x, accelerometer_y, accelerometer_z, datetime_ist, account_number, date`

Still unconfirmed: `bvp`, `systolicPeaks`, `temperature`, `steps` — run the INSPECT cell and fix `COLUMN_MAP` if these differ.

**One-time installs:**
```
pip install pandas numpy scipy agcounts neurokit2
```


Here's the method behind each, as implemented in the notebook:

**PR (Pulse Rate)**
- Method: 60 / IBI, where IBI = interval between consecutive systolic peak timestamps
- Standard basis: this is the conventional IBI→HR conversion used in PPG/BVP heart-rate pipelines (peak-detection framework of Elgendi et al., 2013, *"Systolic Peak Detection in Acceleration Photoplethysmograms"*)

**RESP (Respiration Rate)**
- Method: RIFV — Respiratory-Induced Frequency Variation. The IBI series (from systolic peaks) is resampled to an even time grid, bandpass-filtered to the respiratory band (0.1–0.5 Hz, i.e. 6–30 breaths/min), and peaks in the filtered signal are counted per window
- Standard basis: Charlton et al. 2016/2018 (*Physiological Measurement*), the standard review/benchmark of PPG-derived respiration algorithms (RIAV/RIFV/RIIV family)

**ACT_COUNT**
- Method: `agcounts` package if installed — a published, open-sourced reimplementation of the proprietary ActiGraph counts algorithm (bandpass filter → downsample → rectify/sum per epoch)
- Standard basis: Neishabouri et al. 2022 (*Scientific Reports*), *"Quantification of acceleration as activity counts in ActiGraph wearable"*
- Fallback (if `agcounts` isn't installed): a simplified bandpass (0.25–2.5 Hz) + rectify + sum approximation of the same filter design — noted in the notebook as an approximation, not the exact algorithm

**ACT_INT (activity intensity) & MET**
- Method: ENMO (Euclidean Norm Minus One) computed from raw tri-axial accelerometer, then classified using published wrist cut-points, and MET estimated via a linear regression on ENMO
- Standard basis: Hildebrand et al. 2014 (*Medicine & Science in Sports & Exercise*), the standard wrist-worn ENMO calibration study — cut-points for sedentary/light/moderate/vigorous, and the ENMO→MET regression form

One caveat I flagged in the notebook: the exact MET regression coefficients (`0.972 + 12.24×ENMO`) are an approximation of the Hildebrand form — worth double-checking against the specific published equation variant before citing in a paper, since Hildebrand 2014 reports slightly different coefficients depending on the sub-analysis/age group used.

In [1]:
import os
import glob
import re
import numpy as np
import pandas as pd
from datetime import timedelta
from scipy.signal import butter, filtfilt, find_peaks

BASE_DIR = r"C:\Users\sudip\OneDrive\Desktop\Embrance_user_signals\Embrace_plus_raw\output"

INPUT_MODALITIES = ["accelerometer", "bvp", "eda", "steps", "systolicPeaks", "temperature"]
OUTPUT_MODALITIES = ["PR", "RESP", "ACT_COUNT", "ACT_INT", "MET"]

FILE_EXT = ".csv"
IST_OFFSET = timedelta(hours=5, minutes=30)


In [2]:
def get_user_folders(base_dir=BASE_DIR):
    folders = []
    for name in sorted(os.listdir(base_dir)):
        full = os.path.join(base_dir, name)
        if os.path.isdir(full) and re.match(r"user\d+$", name):
            folders.append((name, full))
    return folders


def get_dates_for_user(user_folder, modality="eda"):
    pattern = os.path.join(user_folder, f"*_{modality}{FILE_EXT}")
    files = glob.glob(pattern)
    dates = []
    for f in files:
        fname = os.path.basename(f)
        m = re.search(r"(\d{4}-\d{2}-\d{2})", fname)
        if m:
            dates.append(m.group(1))
    return sorted(set(dates))


def build_filename(user_folder, user_id, date, modality):
    return os.path.join(user_folder, f"{user_id}_acct0_{date}_{modality}{FILE_EXT}")


def load_modality(user_folder, user_id, date, modality):
    path = build_filename(user_folder, user_id, date, modality)
    if not os.path.exists(path):
        return None
    if FILE_EXT == ".csv":
        return pd.read_csv(path)
    else:
        return pd.read_excel(path)


In [3]:
# ---------------- INSPECT ONE USER / DATE FIRST ----------------
TEST_USER_ID = "user1"
TEST_USER_FOLDER = os.path.join(BASE_DIR, TEST_USER_ID)

dates_found = get_dates_for_user(TEST_USER_FOLDER, modality="eda")
print(f"Dates found for {TEST_USER_ID}: {dates_found}")

if dates_found:
    TEST_DATE = dates_found[0]
    for mod in INPUT_MODALITIES:
        df = load_modality(TEST_USER_FOLDER, TEST_USER_ID, TEST_DATE, mod)
        print(f"\n--- {mod} ---")
        if df is None:
            print("  FILE NOT FOUND")
        else:
            print("  shape:", df.shape)
            print("  columns:", list(df.columns))
            print(df.head(3))


Dates found for user1: ['2026-03-22', '2026-07-13']

--- accelerometer ---
  shape: (347519, 7)
  columns: ['datetime_utc', 'accelerometer_x', 'accelerometer_y', 'accelerometer_z', 'datetime_ist', 'account_number', 'date']
                    datetime_utc  accelerometer_x  accelerometer_y  \
0  2026-03-22 13:35:00.007895750         0.241072        -0.759328   
1  2026-03-22 13:35:00.023520500         0.249856        -0.761280   
2  2026-03-22 13:35:00.039145500         0.256688        -0.765184   

   accelerometer_z                   datetime_ist  account_number        date  
0         0.612928  2026-03-22 19:05:00.007895750               0  2026-03-22  
1         0.610000  2026-03-22 19:05:00.023520500               0  2026-03-22  
2         0.605608  2026-03-22 19:05:00.039145500               0  2026-03-22  

--- bvp ---
  shape: (347390, 5)
  columns: ['datetime_utc', 'bvp', 'datetime_ist', 'account_number', 'date']
                 datetime_utc       bvp                datetime_i

## Column mapping — confirmed for `eda` and `accelerometer`, ASSUMED for the rest
Check `bvp`, `systolicPeaks`, `temperature`, `steps` against the INSPECT output above and fix if needed.

In [4]:
COLUMN_MAP = {
    "accelerometer": {"time": "datetime_utc", "x": "accelerometer_x", "y": "accelerometer_y", "z": "accelerometer_z"},
    "bvp":           {"time": "datetime_utc", "value": "bvp"},
    "eda":           {"time": "datetime_utc", "value": "eda"},
    "steps":         {"time": "datetime_utc", "value": "steps"},
    "systolicPeaks": {"time": "datetime_utc"},
    "temperature":   {"time": "datetime_utc", "value": "temperature"},
}

ACCOUNT_NUMBER = 0   # matches account_number column in your existing files
ACCEL_FS = 64
BVP_FS   = 64


def parse_dt(series):
    """Return actual pandas Timestamps (UTC, naive) for a datetime_utc column."""
    return pd.to_datetime(series)


def make_output_df(window_start_utc, value_col_name, values):
    """
    Build an output dataframe matching your existing file schema:
    datetime_utc, <value_col_name>, datetime_ist, account_number, date
    window_start_utc: array-like of pandas Timestamps (UTC, naive)
    """
    dt_utc = pd.to_datetime(pd.Series(window_start_utc))
    dt_ist = dt_utc + IST_OFFSET
    out = pd.DataFrame({
        "datetime_utc": dt_utc,
        value_col_name: values,
        "datetime_ist": dt_ist,
        "account_number": ACCOUNT_NUMBER,
        "date": dt_utc.dt.strftime("%Y-%m-%d"),
    })
    return out


## Sanity check: sampling rate
Confirms ACCEL_FS / BVP_FS actually match your data.

In [5]:
for mod, fs_guess in [("accelerometer", ACCEL_FS), ("bvp", BVP_FS), ("eda", 4), ("temperature", 1)]:
    df = load_modality(TEST_USER_FOLDER, TEST_USER_ID, TEST_DATE, mod)
    if df is not None and COLUMN_MAP[mod]["time"] in df.columns:
        t = parse_dt(df[COLUMN_MAP[mod]["time"]])
        dt_median = t.diff().dt.total_seconds().median()
        est_fs = 1.0 / dt_median if dt_median and dt_median > 0 else float("nan")
        print(f"{mod}: n={len(df)}, median dt={dt_median:.4f}s, estimated fs={est_fs:.2f} Hz (nominal guess {fs_guess} Hz)")


accelerometer: n=347519, median dt=0.0156s, estimated fs=64.00 Hz (nominal guess 64 Hz)
bvp: n=347390, median dt=0.0156s, estimated fs=64.00 Hz (nominal guess 64 Hz)
eda: n=21720, median dt=0.2500s, estimated fs=4.00 Hz (nominal guess 4 Hz)
temperature: n=5430, median dt=1.0000s, estimated fs=1.00 Hz (nominal guess 1 Hz)


## PR (Pulse Rate) — from systolicPeaks

In [6]:
def compute_pr(systolic_df, window_sec=30):
    """
    PR from consecutive systolic peak timestamps: PR = 60 / IBI (sec).
    Mean PR per non-overlapping window, window_start reported as a real UTC timestamp.
    Reference: standard IBI->HR conversion (Elgendi et al. 2013 peak-detection framework).
    """
    if systolic_df is None or len(systolic_df) < 2:
        return make_output_df([], "PR", [])

    time_col = COLUMN_MAP["systolicPeaks"]["time"]
    t_dt = parse_dt(systolic_df[time_col]).sort_values().reset_index(drop=True)
    t0 = t_dt.iloc[0]
    t_sec = (t_dt - t0).dt.total_seconds().values

    ibi = np.diff(t_sec)
    valid = ibi > 0
    ibi = ibi[valid]
    if len(ibi) == 0:
        return make_output_df([], "PR", [])
    inst_pr = 60.0 / ibi
    peak_sec = t_sec[1:][valid]

    win_idx = (peak_sec // window_sec).astype(int)
    tmp = pd.DataFrame({"win": win_idx, "PR": inst_pr})
    agg = tmp.groupby("win")["PR"].mean().reset_index()
    window_start_utc = [t0 + timedelta(seconds=int(w) * window_sec) for w in agg["win"]]
    return make_output_df(window_start_utc, "PR", agg["PR"].values)


## RESP (Respiration Rate) — from systolic-peak IBI series (RIFV)

In [7]:
def compute_resp(systolic_df, window_sec=30, fs_target=4.0):
    if systolic_df is None or len(systolic_df) < 4:
        return make_output_df([], "RESP", [])

    time_col = COLUMN_MAP["systolicPeaks"]["time"]
    t_dt = parse_dt(systolic_df[time_col]).sort_values().reset_index(drop=True)
    t0 = t_dt.iloc[0]
    t_sec = (t_dt - t0).dt.total_seconds().values

    ibi = np.diff(t_sec)
    peak_sec = t_sec[1:]
    if len(ibi) < 4:
        return make_output_df([], "RESP", [])

    g0, g1 = peak_sec[0], peak_sec[-1]
    if g1 <= g0:
        return make_output_df([], "RESP", [])

    grid = np.arange(g0, g1, 1.0 / fs_target)
    ibi_interp = np.interp(grid, peak_sec, ibi)

    nyq = fs_target / 2.0
    low, high = 0.1 / nyq, min(0.5 / nyq, 0.99)
    b, a = butter(2, [low, high], btype="band")
    filtered = filtfilt(b, a, ibi_interp)

    n_per_win = int(window_sec * fs_target)
    window_start_utc, resp_vals = [], []
    for start in range(0, len(filtered) - n_per_win, n_per_win):
        seg = filtered[start:start + n_per_win]
        peaks, _ = find_peaks(seg, distance=fs_target * 1.5)
        resp_vals.append(len(peaks) * (60.0 / window_sec))
        window_start_utc.append(t0 + timedelta(seconds=grid[start]))

    return make_output_df(window_start_utc, "RESP", resp_vals)


## ACT_COUNT — ActiGraph-style counts from raw accelerometer

In [8]:
try:
    from agcounts.extract import get_counts
    HAVE_AGCOUNTS = True
except ImportError:
    HAVE_AGCOUNTS = False
    print("agcounts not installed -- using fallback ENMO-based approximation. "
          "For true ActiGraph-equivalent counts run: pip install agcounts")


def get_xyz(accel_df):
    cmap = COLUMN_MAP["accelerometer"]
    missing = [c for c in (cmap["x"], cmap["y"], cmap["z"]) if c not in accel_df.columns]
    if missing:
        raise KeyError(
            f"Accelerometer columns {missing} not found. Actual columns: {list(accel_df.columns)}. "
            f"Update COLUMN_MAP['accelerometer'] above."
        )
    return accel_df[[cmap["x"], cmap["y"], cmap["z"]]].values


def compute_act_count(accel_df, window_sec=30, fs=ACCEL_FS):
    if accel_df is None or len(accel_df) < fs:
        return make_output_df([], "ACT_COUNT", [])

    time_col = COLUMN_MAP["accelerometer"]["time"]
    t_dt = parse_dt(accel_df[time_col]).reset_index(drop=True)
    t0 = t_dt.iloc[0]
    xyz = get_xyz(accel_df)

    if HAVE_AGCOUNTS:
        counts = get_counts(xyz, freq=fs, epoch=window_sec)
        counts_vm = np.sqrt((counts ** 2).sum(axis=1))
        window_start_utc = [t0 + timedelta(seconds=i * window_sec) for i in range(len(counts_vm))]
        return make_output_df(window_start_utc, "ACT_COUNT", counts_vm)

    nyq = fs / 2.0
    b, a = butter(4, [0.25 / nyq, min(2.5 / nyq, 0.99)], btype="band")
    vm = np.sqrt((xyz ** 2).sum(axis=1))
    filtered = filtfilt(b, a, vm)
    rectified = np.abs(filtered)

    n_per_win = int(window_sec * fs)
    window_start_utc, vals = [], []
    for start in range(0, len(rectified) - n_per_win, n_per_win):
        vals.append(rectified[start:start + n_per_win].sum())
        window_start_utc.append(t0 + timedelta(seconds=start / fs))
    return make_output_df(window_start_utc, "ACT_COUNT", vals)


agcounts not installed -- using fallback ENMO-based approximation. For true ActiGraph-equivalent counts run: pip install agcounts


## ACT_INT & MET — from ENMO (Hildebrand 2014 wrist cut-points/regression)

**Assumes accelerometer units are in g** (values ~1.0 at rest). Check the INSPECT output and adjust if your data is in mg.

In [9]:
def compute_enmo_series(accel_df, window_sec, fs=ACCEL_FS):
    time_col = COLUMN_MAP["accelerometer"]["time"]
    t_dt = parse_dt(accel_df[time_col]).reset_index(drop=True)
    t0 = t_dt.iloc[0]
    xyz = get_xyz(accel_df)
    vm = np.sqrt((xyz ** 2).sum(axis=1))
    enmo = np.maximum(vm - 1.0, 0.0)   # assumes units of g

    n_per_win = int(window_sec * fs)
    window_start_utc, vals = [], []
    for start in range(0, len(enmo) - n_per_win, n_per_win):
        vals.append(enmo[start:start + n_per_win].mean())
        window_start_utc.append(t0 + timedelta(seconds=start / fs))
    return window_start_utc, np.array(vals)


def compute_act_int_met(accel_df, window_sec=30, fs=ACCEL_FS):
    if accel_df is None or len(accel_df) < fs:
        return make_output_df([], "ACT_INT", []), make_output_df([], "MET", [])

    window_start_utc, enmo = compute_enmo_series(accel_df, window_sec, fs)
    if len(enmo) == 0:
        return make_output_df([], "ACT_INT", []), make_output_df([], "MET", [])

    enmo_mg = enmo * 1000

    def classify(x):
        if x < 47.4:
            return "sedentary"
        elif x < 69.1:
            return "light"
        elif x < 258.7:
            return "moderate"
        else:
            return "vigorous"

    act_int_vals = [classify(x) for x in enmo_mg]
    met_vals = 0.972 + 12.24 * enmo

    act_int_df = make_output_df(window_start_utc, "ACT_INT", act_int_vals)
    met_df = make_output_df(window_start_utc, "MET", met_vals)
    return act_int_df, met_df


## Master per-date processor + save

In [10]:
def process_date(user_id, user_folder, date, window_sec=30, save=True):
    accel_df    = load_modality(user_folder, user_id, date, "accelerometer")
    systolic_df = load_modality(user_folder, user_id, date, "systolicPeaks")

    results = {}
    results["PR"]   = compute_pr(systolic_df, window_sec)
    results["RESP"] = compute_resp(systolic_df, window_sec)

    if accel_df is not None:
        results["ACT_COUNT"] = compute_act_count(accel_df, window_sec)
        act_int_df, met_df = compute_act_int_met(accel_df, window_sec)
        results["ACT_INT"] = act_int_df
        results["MET"] = met_df
    else:
        print(f"  accelerometer file missing for {user_id} {date} -- skipping ACT_COUNT/ACT_INT/MET")

    if save:
        for modality, df in results.items():
            out_path = build_filename(user_folder, user_id, date, modality)
            df.to_csv(out_path, index=False)
            print(f"  saved {out_path}  ({len(df)} rows)")

    return results


## Step 1 — test on a single date for one user

In [11]:
results = process_date(TEST_USER_ID, TEST_USER_FOLDER, TEST_DATE, window_sec=30, save=False)
for mod, df in results.items():
    print(f"\n=== {mod} ===")
    print(df.head())



=== PR ===
                   datetime_utc          PR                  datetime_ist  \
0 2026-03-22 13:35:00.387401396   96.192287 2026-03-22 19:05:00.387401396   
1 2026-03-22 13:35:30.387401396  113.324994 2026-03-22 19:05:30.387401396   
2 2026-03-22 13:36:00.387401396  110.146844 2026-03-22 19:06:00.387401396   
3 2026-03-22 13:36:30.387401396  106.570012 2026-03-22 19:06:30.387401396   
4 2026-03-22 13:37:00.387401396  109.267984 2026-03-22 19:07:00.387401396   

   account_number        date  
0               0  2026-03-22  
1               0  2026-03-22  
2               0  2026-03-22  
3               0  2026-03-22  
4               0  2026-03-22  

=== RESP ===
                   datetime_utc  RESP                  datetime_ist  \
0 2026-03-22 13:35:00.920408396  16.0 2026-03-22 19:05:00.920408396   
1 2026-03-22 13:35:30.920408396  16.0 2026-03-22 19:05:30.920408396   
2 2026-03-22 13:36:00.920408396  20.0 2026-03-22 19:06:00.920408396   
3 2026-03-22 13:36:30.920408396  20

Once the output above looks right (check `datetime_utc` vs `datetime_ist` are 5:30 apart, `account_number`/`date` match your existing files), set `save=True` and run for all dates of this one user:

In [13]:
for date in dates_found:
    print(f"Processing {TEST_USER_ID} / {date} ...")
    process_date(TEST_USER_ID, TEST_USER_FOLDER, date, window_sec=30, save=True)


Processing user1 / 2026-03-22 ...
  saved C:\Users\sudip\OneDrive\Desktop\Embrance_user_signals\Embrace_plus_raw\output\user1\user1_acct0_2026-03-22_PR.csv  (181 rows)
  saved C:\Users\sudip\OneDrive\Desktop\Embrance_user_signals\Embrace_plus_raw\output\user1\user1_acct0_2026-03-22_RESP.csv  (180 rows)
  saved C:\Users\sudip\OneDrive\Desktop\Embrance_user_signals\Embrace_plus_raw\output\user1\user1_acct0_2026-03-22_ACT_COUNT.csv  (180 rows)
  saved C:\Users\sudip\OneDrive\Desktop\Embrance_user_signals\Embrace_plus_raw\output\user1\user1_acct0_2026-03-22_ACT_INT.csv  (180 rows)
  saved C:\Users\sudip\OneDrive\Desktop\Embrance_user_signals\Embrace_plus_raw\output\user1\user1_acct0_2026-03-22_MET.csv  (180 rows)
Processing user1 / 2026-07-13 ...
  saved C:\Users\sudip\OneDrive\Desktop\Embrance_user_signals\Embrace_plus_raw\output\user1\user1_acct0_2026-07-13_PR.csv  (122 rows)
  saved C:\Users\sudip\OneDrive\Desktop\Embrance_user_signals\Embrace_plus_raw\output\user1\user1_acct0_2026-07-1

## Step 2 — once satisfied, scale to all 21 users

In [ ]:
ALL_USERS = get_user_folders(BASE_DIR)
print(f"Found {len(ALL_USERS)} user folders:", [u for u, _ in ALL_USERS])

for user_id, user_folder in ALL_USERS:
    user_dates = get_dates_for_user(user_folder, modality="eda")
    print(f"\n{user_id}: {len(user_dates)} date(s) -> {user_dates}")
    for date in user_dates:
        try:
            process_date(user_id, user_folder, date, window_sec=30, save=True)
        except Exception as e:
            print(f"  !! FAILED {user_id} {date}: {e}")
